In [12]:
from sympy import symbols, Matrix, kronecker_product, solve, sqrt

# Step 1: Base states
q0, q1 = Matrix([1, 0]), Matrix([0, 1])

# Step 2: Alice's unknown 2-qubit input
alpha, beta, gamma, delta = symbols('alpha beta gamma delta')
input_state = alpha * kronecker_product(q0, q0) + \
              beta  * kronecker_product(q0, q1) + \
              gamma * kronecker_product(q1, q0) + \
              delta * kronecker_product(q1, q1)

# Step 3: Channel Template (16x1 column matrix)
c = symbols('c1:17')
unknown_channel = Matrix(c)

# Step 4: Joint 6-qubit system
total_system = kronecker_product(input_state, unknown_channel)

# Step 5: FIXED Bell Measurement
# We want |Phi+> on positions (1,3) and positions (2,4)
bell_single = (1 / sqrt(2)) * (kronecker_product(q0, q0) + kronecker_product(q1, q1))

bell_4qubit = Matrix.zeros(16, 1)
# Correctly permute the tensor product layout so Alice mixes (Input_1, Channel_1) and (Input_2, Channel_2)
for i in range(2):
    for j in range(2):
        for k in range(2):
            for l in range(2):
                # bell_single components for pairs (i, k) and (j, l)
                amp = bell_single[i*2 + k] * bell_single[j*2 + l]
                # Index in Alice's 4-qubit space (i, j, k, l)
                idx = i*8 + j*4 + k*2 + l
                bell_4qubit[idx] = amp

# Step 6: Vectorized Partial Trace
system_matrix = total_system.reshape(16, 4)
bob_leftover = (bell_4qubit.T * system_matrix).T  # 4x1 vector

# Step 7: Target state
target_state = Matrix([alpha, beta, gamma, delta])

# Step 8 & 9: Isolate equations and solve
equations = []
for r in range(4):
    for param in [alpha, beta, gamma, delta]:
        equations.append(bob_leftover[r].coeff(param) - target_state[r].coeff(param))

solution = solve(equations, c)

print("=== 2-Qubit Channel Discovery Complete [Normalization Required] ===")
print(solution)

=== 2-Qubit Channel Discovery Complete [Normalization Required] ===
{c1: 2, c10: 0, c11: 2, c12: 0, c13: 0, c14: 0, c15: 0, c16: 2, c2: 0, c3: 0, c4: 0, c5: 0, c6: 2, c7: 0, c8: 0, c9: 0}
